In [1]:
# Cell 1 — Check Kaggle environment, GPUs, and locate JSONL chunk files
import os
import subprocess
import json
from pathlib import Path

print("=== 1. Python Version ===")
print(subprocess.run(["python", "--version"], capture_output=True, text=True).stdout.strip())

print("\n=== 2. Attached Kaggle Input Folders ===")
input_root = Path("/kaggle/input")
if input_root.exists():
    for p in sorted(input_root.iterdir()):
        print(" -", p)
else:
    print("⚠️ /kaggle/input does not exist. Please attach your dataset.")

print("\n=== 3. GPU Status ===")
gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(gpu_info if gpu_info else "⚠️ No GPU detected. Check Accelerator settings.")

print("\n=== 4. Locating JSONL Chunk Files ===")
# Find all .jsonl files recursively in the input directory
jsonl_files = sorted(list(input_root.rglob("*.jsonl")))
print(f"Found {len(jsonl_files)} JSONL files.")
if len(jsonl_files) == 0:
    print("⚠️ No .jsonl files found. Did you upload the JSONL dataset instead of Markdown?")

if jsonl_files:
    print("\nDirectory structure sample (first 10 files):")
    for f in jsonl_files[:10]:
        print(" -", f.relative_to(input_root))
    if len(jsonl_files) > 10:
        print(f" ... and {len(jsonl_files) - 10} more.")
        
    print("\n=== 5. Previewing the first JSON object ===")
    try:
        with open(jsonl_files[0], 'r', encoding='utf-8') as f:
            first_line = f.readline()
            parsed = json.loads(first_line)
            print(f"File: {jsonl_files[0].name}")
            print(f"Keys available: {list(parsed.keys())}")
            print(f"Chunk ID: {parsed.get('chunk_id')}")
            print(f"Text preview: {parsed.get('text_markdown', '')[:250]}...")
    except Exception as e:
        print(f"Error reading first file: {e}")

=== 1. Python Version ===
Python 3.13.15

=== 2. Attached Kaggle Input Folders ===
 - /kaggle/input/datasets

=== 3. GPU Status ===
Fri Oct 23 05:15:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|       

In [2]:
# Cell 2 — Install/update inference libraries
%pip -q install -U "transformers>=4.51.0" accelerate safetensors bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cell 3 — Configuration
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/medicheck_semantic_labels")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_JSONL = OUTPUT_DIR / "semantic_labels.jsonl"
RESULTS_CSV = OUTPUT_DIR / "semantic_labels.csv"

MODEL_ID = "Qwen/Qwen3-8B"
MAX_ELEMENTS = None  
START_OVER = False

# REDUCED from 6000 to 3000 (approx 750 tokens). 
MAX_INPUT_CHARS = 3000         
NEIGHBOR_CONTEXT_CHARS = 300   
# REDUCED from 2048 to 1536. This fixes the OOM!
MAX_INPUT_TOKENS = 1536        
MAX_NEW_TOKENS = 400
# BATCH SIZE 4 is the sweet spot for T4 memory vs speed
BATCH_SIZE = 4                 

print("Output directory:", OUTPUT_DIR)
print("Batch Size:", BATCH_SIZE)

Output directory: /kaggle/working/medicheck_semantic_labels
Batch Size: 4


In [4]:
# Cell 4 — Load pre-chunked JSONL records and add context
import json
from collections import Counter

all_records = []

# jsonl_files was already created in Cell 1
for file_path in jsonl_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            
            # Map the JSONL fields to what the Qwen prompt expects
            record = {
                "element_id": data.get("chunk_id", "unknown_id"),
                "source_file": data.get("source_file", file_path.name),
                "page_number": data.get("page_start", "Unknown"),
                "element_index": data.get("chunk_index", 0),
                "element_type": data.get("unit_type", "markdown_section"),
                "heading_context": " > ".join(data.get("heading_path", [])),
                "text": data.get("text_markdown", ""),
            }
            all_records.append(record)

# Add neighboring context (context_before and context_after) based on chunk_index
grouped = {}
for r in all_records:
    grouped.setdefault(r["source_file"], []).append(r)

for same_file_records in grouped.values():
    # Sort by chunk index to ensure correct reading order
    same_file_records.sort(key=lambda x: x["element_index"])
    
    for idx, r in enumerate(same_file_records):
        # Get 500 characters of context before and after
        r["context_before"] = same_file_records[idx - 1]["text"][:500] if idx > 0 else ""
        r["context_after"] = same_file_records[idx + 1]["text"][:500] if idx + 1 < len(same_file_records) else ""

print(f"✅ Total elements/chunks loaded: {len(all_records):,}")
print("\n📊 Element types breakdown:")
print(Counter(r["element_type"] for r in all_records))

print("\n👀 Preview of the first 2 loaded elements:")
for item in all_records[:2]:
    print("\n" + "="*50)
    print("ID:", item["element_id"], "| Type:", item["element_type"], "| Page:", item["page_number"])
    print("Heading Path:", item["heading_context"] or "(none)")
    print("--- TEXT PREVIEW ---")
    print(item["text"][:500] + ("..." if len(item["text"]) > 500 else ""))
    print("="*50)

✅ Total elements/chunks loaded: 3,029

📊 Element types breakdown:
Counter({'markdown_section': 2724, 'document_content': 305})

👀 Preview of the first 2 loaded elements:

ID: DOC_f3d1085ecf29_C0000 | Type: markdown_section | Page: None
Heading Path: <u>Documents to be submitted for grant of permission to conduct Bioequivalence studies for export purpose.</u>
--- TEXT PREVIEW ---
# <u>Documents to be submitted for grant of permission to conduct Bioequivalence studies for export purpose.</u>

A large number of applications are being filed to the office of DCG (I) at CDSCO (HQ) by Pharmaceutical companies, both manufacturers and importers as well as CROs on behalf of them,  requesting for the approval to carry out BE studies with various pharmaceutical dosage formulations  on Indian subjects.

In light of the above, for easy processing of such applications and to bring uni...

ID: DOC_f3d1085ecf29_C0001 | Type: markdown_section | Page: None
Heading Path: <u>Documents to be submitted for g

In [5]:
# Cell 5 — Load Qwen3-8B across 2 GPUs (FP16)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is available.")

num_gpus = torch.cuda.device_count()
print(f"CUDA GPUs detected: {num_gpus}")

max_memory = {gpu_idx: "13GiB" for gpu_idx in range(num_gpus)}
max_memory["cpu"] = "24GiB"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading model in FP16 across GPUs...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
model.eval()
print("\n✅ Model ready.")

CUDA GPUs detected: 2
Loading tokenizer...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


✅ Model ready.


In [6]:
# Cell 6 — Define the grounded semantic-labeling prompt
SYSTEM_PROMPT = """You are labeling excerpts from a corpus of pharmaceutical and drug-regulatory documents.
Your task is semantic labeling of ONE supplied Markdown element, not general question-answering.

Rules:
- Describe what this exact element contains or does. Use a concise, specific, human-readable semantic_label.
- This is exploratory corpus profiling: do not force the element into a predefined taxonomy.
- Ground every statement in the supplied element and limited context. Do not add outside facts.
- A medicine being mentioned does NOT by itself mean it is banned, recalled, unsafe, counterfeit, approved, or genuine.
- Distinguish draft/proposed material from final/effective material only when the excerpt explicitly supports that distinction.
- If legal effect or regulatory status cannot be established from this excerpt, say so; do not guess.
- For tables, interpret the caption/title, headers, and rows together. If a table is split or malformed, describe what is actually visible and note uncertainty.
- evidence_quote must be an exact short quotation copied from the element's text, not from context. If no concise supporting quote is suitable, use an empty string.
- Do not generate, alter, or guess element IDs, filenames, page numbers, dates, law sections, or citations.
- Return exactly one valid JSON object and no Markdown fences or extra commentary."""

USER_TEMPLATE = """Label the following single document element.

METADATA (provided by the pipeline; do not modify):
element_type: {element_type}
heading_context: {heading_context}
page_number: {page_number}

NEARBY CONTEXT BEFORE (for interpretation only; do not cite as the element's evidence):
{context_before}

TARGET ELEMENT (this is the content to label and quote from):
{target_text}

NEARBY CONTEXT AFTER (for interpretation only; do not cite as the element's evidence):
{context_after}

Return this JSON schema, with all keys present:
{{
  "semantic_label": "one concise, descriptive label for this element",
  "summary": "faithful summary in 1–2 sentences",
  "subject": "main medicine, ingredient, company, authority, document topic, or other subject; say unclear if not identifiable",
  "regulatory_meaning": "what the excerpt explicitly communicates or does, or state that the meaning is not determinable from this excerpt",
  "legal_status": "exactly one of: final_or_effective_explicitly_stated; draft_or_proposed_explicitly_stated; administrative_or_informational; unclear_from_excerpt; not_applicable",
  "evidence_quote": "an exact short quote from TARGET ELEMENT, or empty string",
  "uncertainties": ["brief uncertainties, ambiguities, or extraction issues; empty list if none obvious"]
}}
"""
print("✅ Prompt configured.")

✅ Prompt configured.


In [7]:
# Cell 7 — Helper functions for prompt construction, parsing, and resume
import re
import json
from tqdm.auto import tqdm
from typing import Dict, Optional, List

def clip_text(s: Optional[str], limit: int) -> str:
    s = (s or "").strip()
    if len(s) <= limit:
        return s
    return s[:limit] + "\n[TRUNCATED FOR MODEL INPUT]"

def build_prompt(r: Dict) -> str:
    target = clip_text(r["text"], MAX_INPUT_CHARS)
    user_msg = USER_TEMPLATE.format(
        element_type=r.get("element_type", "unknown"),
        heading_context=r.get("heading_context") or "(none)",
        page_number=r.get("page_number") if r.get("page_number") else "(not marked)",
        context_before=clip_text(r.get("context_before", ""), NEIGHBOR_CONTEXT_CHARS),
        target_text=target,
        context_after=clip_text(r.get("context_after", ""), NEIGHBOR_CONTEXT_CHARS),
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg + "\n/no_think"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

def extract_first_json(text: str):
    """Extract the first complete JSON object while respecting braces inside strings."""
    if not text: return None
    cleaned = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text.strip(), flags=re.I)
    start = cleaned.find("{")
    if start < 0: return None
    in_string = escaped = False
    depth = 0
    for i in range(start, len(cleaned)):
        ch = cleaned[i]
        if in_string:
            if escaped: escaped = False
            elif ch == "\\": escaped = True
            elif ch == '"': in_string = False
        else:
            if ch == '"': in_string = True
            elif ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try: return json.loads(cleaned[start:i + 1])
                    except json.JSONDecodeError: return None
    return None

def normalize_model_result(parsed, raw_text: str) -> Dict:
    expected = ["semantic_label", "summary", "subject", "regulatory_meaning", "legal_status", "evidence_quote", "uncertainties"]
    if not isinstance(parsed, dict):
        return {
            "semantic_label": "", "summary": "", "subject": "", "regulatory_meaning": "",
            "legal_status": "unclear_from_excerpt", "evidence_quote": "",
            "uncertainties": ["Could not parse model output as valid JSON; inspect model_raw_output."],
            "parse_status": "parse_error", "model_raw_output": raw_text,
        }
    out = {k: parsed.get(k) for k in expected}
    for key in ["semantic_label", "summary", "subject", "regulatory_meaning", "legal_status", "evidence_quote"]:
        if not isinstance(out.get(key), str):
            out[key] = "" if out.get(key) is None else str(out.get(key))
    if not isinstance(out.get("uncertainties"), list):
        out["uncertainties"] = [str(out["uncertainties"])] if out.get("uncertainties") else []
    out["parse_status"] = "ok"
    out["model_raw_output"] = raw_text
    
    legal_values = {"final_or_effective_explicitly_stated", "draft_or_proposed_explicitly_stated", "administrative_or_informational", "unclear_from_excerpt", "not_applicable"}
    if out["legal_status"] not in legal_values:
        out["uncertainties"].append("Model returned a legal_status value outside the requested options; review manually.")
    return out

def read_existing_results(path):
    existing = {}
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        obj = json.loads(line)
                        if obj.get("element_id"): existing[obj["element_id"]] = obj
                    except json.JSONDecodeError: pass
    return existing

if START_OVER and RESULTS_JSONL.exists():
    RESULTS_JSONL.unlink()
existing_results = read_existing_results(RESULTS_JSONL)
print(f"✅ Existing completed elements available for resume: {len(existing_results):,}")

✅ Existing completed elements available for resume: 0


In [8]:
# Cell 8 — Generate labels and checkpoint after each batch
import time

records_to_process = all_records if MAX_ELEMENTS is None else all_records[:MAX_ELEMENTS]
pending = [r for r in records_to_process if r["element_id"] not in existing_results]
print(f"Selected elements: {len(records_to_process):,}")
print(f"Already complete: {len(records_to_process) - len(pending):,}")
print(f"Remaining in this run: {len(pending):,}")

input_device = model.get_input_embeddings().weight.device
print("Input embedding device:", input_device)

def generate_batch(batch: List[Dict]) -> List[Dict]:
    prompts = [build_prompt(r) for r in batch]
    encoded = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_TOKENS)
    encoded = {k: v.to(input_device) for k, v in encoded.items()}
    prompt_length = encoded["input_ids"].shape[1]
    
    with torch.inference_mode():
        generated = model.generate(
            **encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            repetition_penalty=1.03, use_cache=True, pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        
    answer_tokens = generated[:, prompt_length:]
    raw_answers = tokenizer.batch_decode(answer_tokens, skip_special_tokens=True)
    
    output_rows = []
    for r, raw in zip(batch, raw_answers):
        parsed = extract_first_json(raw)
        semantic = normalize_model_result(parsed, raw)
        
        row = {
            "element_id": r["element_id"],
            "source_file": r["source_file"],
            "page_number": r["page_number"],
            "element_index": r["element_index"],
            "element_type": r["element_type"],
            "heading_context": r["heading_context"],
            "original_text": r["text"],
            **semantic,
            "model_id": MODEL_ID,
        }
        output_rows.append(row)
    return output_rows

processed_this_run = 0
start_time = time.time()

with RESULTS_JSONL.open("a", encoding="utf-8") as out_file:
    for start in tqdm(range(0, len(pending), BATCH_SIZE), desc="Semantic labeling"):
        batch = pending[start:start + BATCH_SIZE]
        try:
            batch_results = generate_batch(batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(batch) > 1:
                print("\n⚠️ OOM error. Retrying batch one element at a time.")
                batch_results = []
                for item in batch:
                    batch_results.extend(generate_batch([item]))
            else:
                raise
                
        for result in batch_results:
            out_file.write(json.dumps(result, ensure_ascii=False) + "\n")
            existing_results[result["element_id"]] = result
            
        out_file.flush() 
        processed_this_run += len(batch_results)

print(f"\n✅ Completed in {time.time() - start_time:.1f} seconds.")
print(f"Newly processed: {processed_this_run:,}")
print(f"JSONL checkpoint saved to: {RESULTS_JSONL}")

Selected elements: 3,029
Already complete: 0
Remaining in this run: 3,029
Input embedding device: cuda:0


Semantic labeling:   0%|          | 0/758 [00:00<?, ?it/s]


✅ Completed in 21673.1 seconds.
Newly processed: 3,029
JSONL checkpoint saved to: /kaggle/working/medicheck_semantic_labels/semantic_labels.jsonl


In [9]:
# Cell 9 — Export checkpointed results to CSV and display for review
import pandas as pd
import json
from IPython.display import display

all_results = read_existing_results(RESULTS_JSONL)
result_rows = list(all_results.values())

if not result_rows:
    raise RuntimeError("No results found. Did Cell 8 finish?")

# Make lists (like uncertainties) CSV-friendly
for row in result_rows:
    if isinstance(row.get("uncertainties"), list):
        row["uncertainties"] = " | ".join(map(str, row["uncertainties"]))

results_df = pd.DataFrame(result_rows)

# Reorder columns for easy reading
preferred_cols = [
    "element_id", "semantic_label", "subject", "legal_status", 
    "summary", "regulatory_meaning", "evidence_quote", "parse_status"
]
cols_to_show = [c for c in preferred_cols if c in results_df.columns]
display_cols = [c for c in cols_to_show if c not in ["evidence_quote", "regulatory_meaning"]]

print(f"✅ Total rows: {len(results_df):,}")
print("\n👀 Preview of the first 10 semantic labels:")
# Increase column width so we can read the text
pd.set_option('display.max_colwidth', 150)
display(results_df[display_cols].head(10))

# Save full CSV
results_df.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
print(f"\n💾 Full CSV saved to: {RESULTS_CSV}")

✅ Total rows: 3,029

👀 Preview of the first 10 semantic labels:


,element_id,semantic_label,subject,legal_status,summary,parse_status
0,DOC_f3d1085ecf29_C0000,Guidelines for submitting documents for bioequivalence study permissions for export,Bioequivalence studies for export,administrative_or_informational,The text outlines the need for standardized submission of documents by pharmaceutical companies and CROs seeking approval to conduct bioequivalenc...,ok
1,DOC_f3d1085ecf29_C0001,List of requirements for conducting bioequivalence studies for export of a new molecule approved abroad but not in India,Bioequivalence study requirements for a new molecule not approved in India but approved elsewhere,administrative_or_informational,This section outlines the specific documents and data required to apply for permission to conduct bioequivalence studies for a new molecule that i...,ok
2,DOC_f3d1085ecf29_C0002,List of required documents for Bioequivalence studies of New Drugs approved in India within 1 year,New Drugs approved in India within 1 year,administrative_or_informational,"This section outlines the documents required to be submitted for granting permission to conduct Bioequivalence studies for export purposes, specif...",ok
3,DOC_f3d1085ecf29_C0003,List of documents required for granting permission to conduct bioequivalence studies for export,Bioequivalence studies for export,administrative_or_informational,"This section outlines the specific documents required for obtaining permission to conduct bioequivalence studies for export, applicable to new dru...",ok
4,DOC_9d706ff061d8_C0000,Revised checklist for BA/BE NOC (draft for comments),BA/BE NOC checklist for drug studies,draft_or_proposed_explicitly_stated,"This element presents a revised checklist for obtaining a BA/BE NOC, indicating it is a draft version intended for public comment prior to a speci...",ok
5,DOC_9d706ff061d8_C0001,Overview of documents required for BE study permission in India,Regulatory requirements for bioequivalence studies in India,administrative_or_informational,The document outlines the purpose and scope of documents needed for granting permission to conduct bioequivalence (BE) studies involving human sub...,ok
6,DOC_9d706ff061d8_C0002,List of requirements for conducting bioequivalence studies of unapproved drugs in India,Bioequivalence study requirements for unapproved drugs in India,administrative_or_informational,This section outlines the necessary documents and procedures required to obtain permission for conducting bioequivalence (BE) studies of a new mol...,ok
7,DOC_9d706ff061d8_C0003,List of required documents for conducting BA/BE studies on new drugs approved in India within one year,New drugs approved in India within one year,administrative_or_informational,This section outlines the documentation required for obtaining permission to conduct bioavailability/bioequivalence (BA/BE) studies on new drugs a...,ok
8,DOC_9d706ff061d8_C0004,List of required documents for BA/BE studies in human subjects/patients for export purposes for new drugs approved between 1 and 4 years in India,New drugs approved in India between 1 and 4 years ago,administrative_or_informational,This section outlines the documents required for granting permission to conduct bioavailability/bioequivalence (BA/BE) studies in human subjects o...,ok
9,DOC_9d706ff061d8_C0005,List of required documents for BE NOC for modified release drug products,BE NOC for modified release drug products,administrative_or_informational,This section outlines the documents needed to obtain a Bioequivalence No Objection Certificate (BE NOC) for drug products in modified release form...,ok



💾 Full CSV saved to: /kaggle/working/medicheck_semantic_labels/semantic_labels.csv


In [10]:
# Cell 10 — Quality checks
print("=== Parse Status Counts ===")
parse_status_counts = results_df.get("parse_status", pd.Series(dtype=str)).value_counts(dropna=False)
print(parse_status_counts)

empty_label_count = results_df.get("semantic_label", pd.Series(dtype=str)).fillna("").str.strip().eq("").sum()
print(f"\n⚠️ Empty semantic labels: {empty_label_count:,} / {len(results_df):,}")

print("\n=== Legal Status Distribution ===")
if "legal_status" in results_df.columns:
    print(results_df["legal_status"].value_counts(dropna=False))

# Show any rows that failed to parse or have an empty label
print("\n=== Examples needing manual review (parse failures or empty labels) ===")
review_mask = (
    results_df.get("parse_status", pd.Series(index=results_df.index, dtype=str)).ne("ok") |
    results_df.get("semantic_label", pd.Series(index=results_df.index, dtype=str)).fillna("").str.strip().eq("")
)
review_cols = [c for c in ["element_id", "semantic_label", "original_text", "model_raw_output"] if c in results_df.columns]
if review_mask.any():
    display(results_df.loc[review_mask, review_cols].head(5))
else:
    print("🎉 Perfect! No parse errors or empty labels in this batch.")

=== Parse Status Counts ===
parse_status
ok             3026
parse_error       3
Name: count, dtype: int64

⚠️ Empty semantic labels: 4 / 3,029

=== Legal Status Distribution ===
legal_status
administrative_or_informational                            1370
final_or_effective_explicitly_stated                       1321
draft_or_proposed_explicitly_stated                         313
                                                             13
unclear_from_excerpt                                          6
banned                                                        3
Prohibited in the public interest with immediate effect       2
not_enough_information                                        1
Name: count, dtype: int64

=== Examples needing manual review (parse failures or empty labels) ===


,element_id,semantic_label,original_text,model_raw_output
1755,DOC_fa2ac0b61a14_C0201,,"And whereas, the Expert Committee earlier appointed by the Central Government also recommended that the said drug is found to have no therapeutic ...","""metadata"": {\n ""element_type"": ""string"",\n ""heading_context"": ""string"",\n ""page_number"": ""string""\n }\n}\n</think>\n\n{\n ""semantic_..."
2337,DOC_e895955f8d1b_C0003,,"# **NOTIFICATION**\n\nNew Delhi, the 25th June, 2021\n\n**S.O. 2572(E).** In pursuance of the powers conferred under sub-rule (2) of rule 18 of th...","New Delhi on 25th June, 2021\n\n<u>[PART IISEC. 3(ii)]</u>\n\n<u>[PART IISEC. 3(ii)]</u>\n\n<u>[PART IISEC. 3(ii)]</u>\n\n<u>[PART IISEC. 3(ii)]<..."
2385,DOC_227d80859513_C0005,,"## x`g ea=ky;\n\nubZ fnYyh] 28 Qjojh] 2018\n\n1054 GI/2018\n\n(507)\n\n532\n\nTHE GAZETTE OF INDIA : MARCH 10, 2018/PHALGUNA 19, 1939 [PART IISEC...",Shri. D. Suresh\n\n33. Shri. D. Venkatesh\n\n34. Shri. D. Vijayakumar\n\n35. Shri. D. Vijay Kumar\n\n36. Shri. D. Vijay Kumar\n\n37. Shri. D. Vij...
2893,DOC_2ed7dc604d99_C0009,,"# **<u>Gujarat:</u>**\n\n21. M/s Aarya Lifesciences PVT. LTD, FF/17, ORCHID COMPLEX, SHILAJ ROAD, NR TALAV THALTEJ, AHMEDABAD, 380052, Tal.:Ahmeda...","M/s S. S. Patel & Co. PVT. LTD; Ahmedabad, GUJARAT.\n\n49. M/s S. S. Patel & Co. PVT. LTD; Ahmedabad, GUJARAT.\n\n50. M/s S. S. Patel & Co. PVT. ..."
